# 15-2 Longest palindrome subsequence
A palindrome is a nonempty string over some alphabet that reads the same forward and backward. Examples of palindromes are all strings of length 1, civic, racecar, and aibohphobia (fear of palindromes).
Give an efficient algorithm to find the longest palindrome that is a subsequence of a given input string. For example, given the input character, your algorithm should return carac. What is the running time of your algorithm?

In [ ]:
from fontTools.merge.util import current_time


#Problem 15-2
def palindrome(s: str) -> str:
    if not s:
        return ""

    n = len(s)
    dp = [[0] * n for _ in range(n)]

    for i in range(n):
        dp[i][i] = 1

    for i in range(n-1, -1, -1):
        for j in range(i+1, n):
            if s[i] == s[j]:
                dp[i][j] = dp[i+1][j-1] + 2
            else:
                dp[i][j] = max(dp[i+1][j], dp[i][j-1])

    for k in range(n):
        print(dp[k])

    left = []
    right = []
    i, j = 0, n - 1

    while i <= j:
        if i == j:
            left.append(s[i])
            break
        if s[i] == s[j]:
            left.append(s[i])
            right.append(s[j])
            i += 1
            j -= 1
        elif dp[i + 1][j] >= dp[i][j - 1]:
            i += 1
        else:
            j -= 1

    return "".join(left + right[::-1])

15-2:Complexity: Theta(n**2) for the complexity of time and space.

# 15-4 Printing neatly
Consider the problem of neatly printing a paragraph with a monospaced font (all characters having the same width) on a printer. The input text is a sequence of n words of lengths l1​,l2​,…,ln​, measured in characters. We want to print this paragraph neatly on a number of lines that hold a maximum of M characters each. Our criterion of "neatness" is as follows. If a given line contains words i through j, where i≤j, and we leave exactly one space between words, the number of extra space characters at the end of the line is M−j+i−∑k=ij​lk​, which must be nonnegative so that the words fit on the line. We wish to minimize the sum, over all lines except the last, of the cubes of the numbers of extra space characters at the ends of lines. Give a dynamic-programming algorithm to print a paragraph of n words neatly on a printer. Analyze the running time and space requirements of your algorithm.

In [ ]:
def Printing_Neatly(l, n, M):
    Ext = [[0 for i in range(n + 1)] for j in range(n + 1)]
    LCost = [[0 for i in range(n + 1)] for j in range(n + 1)]
    for i in range(1, n + 1):
        Ext[i][i] = M - l[i - 1]
        for j in range(i + 1, n + 1):
            Ext[i][j] = Ext[i][j - 1] - l[j - 1] - 1  # compute extra space
    for i in range(1, n + 1):
        for j in range(i, n + 1):
            if Ext[i][j] < 0:
                LCost[i][j] = sys.maxsize  # this line don’t fit, to set the cost is infinity
            elif Ext[i][j] >= 0 and j == n:
                LCost[i][j] = 0
            else:
                LCost[i][j] = (Ext[i][j]) ^ 3
    C = [0 for i in range(n + 1)]
    R = [0 for i in range(n + 1)]
    for j in range(1, n + 1):
        C[j] = sys.maxsize
        for i in range(1, j + 1):
            if C[i - 1] != sys.maxsize and (LCost[i][j] + C[i - 1]) < C[j]:
                C[j] = LCost[i][j] + C[i - 1]
                R[j] = i

    print_lines(R, n)
    return


def print_lines(R, j):
    no = 0
    i = R[j]
    if i == 1:
        no = 1
    else:
        no = print_lines(R, i - 1) + 1
    print("Line number:", no, end=", ")
    for t in range(i, j):
        print("", word[t - 1], end=" ")
    print("")
    return no  # 行号

15-4:Complexity: Theta(n**2) for time and space complexity

# 15-6 Planning a company party
Professor Stewart is consulting for the president of a corporation that is planning a company party. The company has a hierarchical structure; that is, the supervisor relation forms a tree rooted at the president. The personnel office has ranked each employee with a conviviality rating, which is a real number. In order to make the party fun for all attendees, the president does not want both an employee and his or her immediate supervisor to attend.
Professor Stewart is given the tree that describes the structure of the corporation, using the left-child, right-sibling representation described in Section 10.4. Each node of the tree holds, in addition to the pointers, the name of an employee and that employee's conviviality ranking. Describe an algorithm to make up a guest list that maximizes the sum of the conviviality ratings of the guests. Analyze the running time of your algorithm.

In [ ]:
class Employee(object):
    def __init__(self, name, conviviality):
        self.name = name
        self.conviviality = conviviality
        self.left_child = None
        self.right_sibling = None


def add_children(parent, children):
    """Connect children using left-child, right-sibling representation."""
    if not children:
        return
    parent.left_child = children[0]
    for i in range(len(children) - 1):
        children[i].right_sibling = children[i + 1]


def iter_children(node):
    child = node.left_child
    while child is not None:
        yield child
        child = child.right_sibling


def best_party(node):
    """
    Returns two states for each subtree rooted at node:
    1. include_state: best score/list when node attends.
    2. exclude_state: best score/list when node does not attend.
    """
    if node is None:
        return (0, []), (0, [])

    include_score = node.conviviality
    include_list = [node.name]
    exclude_score = 0
    exclude_list = []

    for child in iter_children(node):
        child_include, child_exclude = best_party(child)

        include_score += child_exclude[0]
        include_list += child_exclude[1]

        if child_include[0] > child_exclude[0]:
            exclude_score += child_include[0]
            exclude_list += child_include[1]
        else:
            exclude_score += child_exclude[0]
            exclude_list += child_exclude[1]

    return (include_score, include_list), (exclude_score, exclude_list)


def make_guest_list(president):
    include_state, exclude_state = best_party(president)
    if include_state[0] > exclude_state[0]:
        return include_state
    return exclude_state

15-6: Theta(n) for time and space complexity

# 16-2 Scheduling to minimize average completion time
Suppose you are given a set S={a1,a2,…,an} of tasks, where task ai requires pi units of processing time to complete, once it has started. You have one computer on which to run these tasks, and the computer can run only one task at a time. Let ci be the completion time of task ai, that is, the time at which task ai completes processing. Your goal is to minimize the average completion time, that is, to minimize (1/n)∑i=1/n ci. For example, suppose there are two tasks, a1 and a2, with p1 =3 and p2 =5, and consider the schedule in which a2 runs first, followed by a1. Then c2 =5, c1 =8, and the average completion time is (5+8)/2=6.5. If task a1 runs first, however, then c1 =3, c2 =8, and the average completion time is (3+8)/2=5.5.

a. Give an algorithm that schedules the tasks so as to minimize the average completion time. Each task must run non-preemptively, that is, once task ai starts, it must run continuously for pi units of time. Prove that your algorithm minimizes the average completion time, and state the running time of your algorithm.

b. Suppose now that the tasks are not all available at once. That is, each task cannot start until its release time ri. Suppose also that we allow preemption, so that a task can be suspended and restarted at a later time. For example, a task ai with processing time pi =6
 and release time ri =1 might start running at time 1 and be preempted at time 4. It might then resume at time 10 but be preempted at time 11, and it might finally resume at time 13 and complete at time 15. Task ai has run for a total of 6 units, but its running time has been divided into three pieces. In this scenario, ai's completion time is 15. Give an algorithm that schedules the tasks so as to minimize the average completion time in this new scenario. Prove that your algorithm minimizes the average completion time, and state the running time of your algorithm.

In [ ]:
# Problem A
def schedule_min_per_task(tasks):
    tasks = sorted(tasks, key=lambda x: x[1])
    current_time = 0
    total_time = 0

    for _, time in tasks:
        current_time += time
        total_time += current_time

    return total_time/len(tasks)

Time Complexity: Theta(nlogn)
Space Complexity: Theta(n)

In [ ]:
# Problem B
import heapq

def schedule_SRPT(tasks):
    tasks.sort(key = lambda x: x[1])
    heap = []
    current_time = 0
    i = 0
    n = len(tasks)
    completion = {}

    while len(completion) < n:
        while i < n and tasks[i][2] <= current_time:
            tid, p, r = tasks[i]
            heapq.heappush(heap, (p, tid))
            i += 1

        if not heap:
            current_time = tasks[i][2]
            continue

        rem, tid = heapq.heappop(heap)
        current_time += rem
        completion[tid] = current_time

    return completion

Time Complexity: Theta(nlogn)
Space Complexity: Theta(n)

In [ ]:
if __name__ == '__main__':
    # Problem 15-2
    print(palindrome('character'))

    # Problem 15-4
    word = ["optimal", "substructure", "in", "the", "following", "way"]
    n = len(word)
    l = [0 for i in range(n)]
    for i in range(n):
        l[i] = len(word[i])
    Printing_Neatly(l, n, 20)